# Vision Transformer (ViT) on MNIST — 快速版 / Fast Demo

> ⚡ **快速版說明 / Fast Demo Notes:**
> - 訓練資料：10,000 張（原版 60,000 張）/ Training data: 10k samples (vs 60k full)
> - 模型：`dim=64, depth=2`（原版 `dim=128, depth=6`）/ Smaller model
> - 預計訓練時間：**CPU ~1-2 分鐘 / ~1-2 min on CPU**
> - 預期準確率：~96–97%（原版 ~98–99%）/ Expected accuracy: ~96-97%

---

## 核心概念 / Core Idea

> 把圖像切成小塊（patch），每塊當作一個「詞元（token）」，然後用 Transformer 處理這個詞元序列。  
> Split an image into patches, treat each patch like a word token, and process the sequence with a Transformer.

```
圖像 (28×28)  →  切成16塊  →  每塊線性嵌入  →  Transformer  →  分類結果
Image          16 patches    Embed patches    Transformer     Class label
```

## 環境安裝 / Setup

In [ ]:
%pip install einops --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from einops import rearrange, repeat
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.metrics import confusion_matrix
import itertools

plt.rcParams['font.family'] = ['Microsoft JhengHei', 'MingLiU', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用裝置 / Device: {device}')
print(f'PyTorch 版本 / Version: {torch.__version__}')

---
## 第一步：載入 MNIST 資料（子集）/ Step 1 — Load MNIST (Subset)

**中文：** 只取訓練集前 10,000 張加速演示，測試集保留全部 10,000 張評估準確率。  
**EN:** Use only the first 10,000 training images for speed. Full 10k test set is kept for accurate evaluation.

In [ ]:
IMAGE_SIZE  = 28
PATCH_SIZE  = 7
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 4×4 = 16
PATCH_DIM   = PATCH_SIZE * PATCH_SIZE * 1      # 7×7×1 = 49
SUBSET_SIZE = 10000                             # ⚡ 只用 10k 張

print('── 關鍵參數 / Key Parameters ──')
print(f'圖像尺寸   Image size   : {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'塊尺寸     Patch size   : {PATCH_SIZE}×{PATCH_SIZE}')
print(f'塊數量     Num patches  : {NUM_PATCHES}')
print(f'訓練子集   Subset size  : {SUBSET_SIZE:,}  ⚡ 加速用')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

# ⚡ 只取前 10,000 張訓練資料 / Use only first 10k training samples
train_data = Subset(full_train, range(SUBSET_SIZE))

train_loader = DataLoader(train_data, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n訓練集 Train: {len(train_data):,} 張  ⚡')
print(f'測試集 Test:  {len(test_data):,} 張')

### 視覺化圖像塊 / Visualize Patches

**中文：** 每個彩色方格就是 Transformer 將要處理的一個「詞元（token）」。  
**EN:** Each colored cell is one token the Transformer will process.

In [ ]:
sample_img, sample_label = full_train[0]
img_np = sample_img.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(img_np, cmap='gray')
axes[0].set_title(f'原始圖像 / Original\n標籤 Label: {sample_label}', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_np, cmap='gray')
n = IMAGE_SIZE // PATCH_SIZE
colors = plt.cm.tab20(np.linspace(0, 1, NUM_PATCHES))
for i in range(n):
    for j in range(n):
        idx = i * n + j
        rect = mpatches.Rectangle(
            (j * PATCH_SIZE - 0.5, i * PATCH_SIZE - 0.5),
            PATCH_SIZE, PATCH_SIZE,
            linewidth=2, edgecolor=colors[idx], facecolor='none'
        )
        axes[1].add_patch(rect)
        axes[1].text(
            j * PATCH_SIZE + PATCH_SIZE // 2 - 0.5,
            i * PATCH_SIZE + PATCH_SIZE // 2,
            str(idx), color=colors[idx], fontsize=9, fontweight='bold', ha='center'
        )
axes[1].set_title(f'切分為 {NUM_PATCHES} 個塊 / {NUM_PATCHES} patches\n每塊 = 1 個 token', fontsize=11)
axes[1].axis('off')

plt.suptitle('把圖像變成詞元序列 / Convert Image into Token Sequence', fontsize=13)
plt.tight_layout()
plt.show()

---
## 第二步：塊嵌入 / Step 2 — Patch Embedding

**中文：** 每個塊（49像素）→ Linear(49→dim) → 嵌入向量。序列開頭加入可學習的 [CLS] 詞元。  
**EN:** Each patch (49 pixels) → Linear(49→dim). Prepend a learnable [CLS] token for classification.

```
patch（49維）→ Linear(49, dim) → embedding（dim維）
```

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_dim, dim):
        super().__init__()
        self.projection = nn.Linear(patch_dim, dim)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, dim))

    def forward(self, x):
        patches = rearrange(x, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                             p1=PATCH_SIZE, p2=PATCH_SIZE)
        tokens = self.projection(patches)
        cls    = repeat(self.cls_token, '1 1 d -> b 1 d', b=x.shape[0])
        return torch.cat([cls, tokens], dim=1)

demo_embed = PatchEmbedding(patch_dim=PATCH_DIM, dim=64)
demo_out   = demo_embed(torch.randn(4, 1, 28, 28))
print(f'塊嵌入輸出 / Output: {demo_out.shape}  (batch=4, 16塊+1CLS, dim=64)')

---
## 第三步：多頭自注意力 / Step 3 — Multi-Head Self-Attention

**中文：** 每個詞元產生 Q/K/V 三個向量，透過縮放點積注意力讓詞元互相「溝通」。  
**EN:** Each token generates Q/K/V. Scaled dot-product attention lets tokens communicate.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, heads, dropout=0.0):
        super().__init__()
        assert dim % heads == 0
        self.heads    = heads
        self.dim_head = dim // heads
        self.scale    = self.dim_head ** -0.5
        self.to_qkv   = nn.Linear(dim, dim * 3, bias=False)
        self.to_out   = nn.Sequential(nn.Linear(dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        b, n, _ = x.shape
        h = self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)
        dots = torch.einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        attn = dots.softmax(dim=-1)
        out  = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        out  = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out), attn

mha = MultiHeadAttention(dim=64, heads=4)
x   = torch.randn(2, 17, 64)
out, attn_w = mha(x)
print(f'輸入 Input  : {x.shape}')
print(f'輸出 Output : {out.shape}')
print(f'注意力矩陣  : {attn_w.shape}  (batch, heads, tokens, tokens)')

### 🔍 Attention Matrix Visualization

**What is the attention matrix?**

For each head, every token produces a **Query (Q)** and a **Key (K)**.  
We compute a score between every pair of tokens:

```
score(i, j) = dot(Q_i, K_j) / sqrt(d_k)   ← "how much should token i attend to token j?"
```

Then apply **softmax row-wise** so each row sums to 1.0 — giving us attention *probabilities*.

```
         attends to →
         CLS  P0   P1  ...  P15
CLS    [ 0.10 0.08 0.06 ... 0.05 ]  ← how much CLS looks at each patch
P0     [ 0.12 0.15 0.03 ... 0.02 ]  ← how much patch 0 looks at others
P1     [ 0.07 0.04 0.20 ... 0.03 ]
...
P15    [ 0.09 0.02 0.04 ... 0.18 ]
```

- **Bright cell (i, j)** = token i pays a lot of attention to token j  
- **CLS row** is the most important — it decides what to look at before classification  
- Each head specializes: one might focus on nearby patches, another on global structure

In [ ]:
# ── Attention Matrix Visualization ──────────────────────────────────────────
# Use a real MNIST image so the attention is more meaningful than random noise
sample_img, sample_label = full_train[0]
sample_tensor = sample_img.unsqueeze(0)  # (1, 1, 28, 28)

# Run through patch embedding to get token sequence, then through MHA
demo_embed_vis = PatchEmbedding(patch_dim=PATCH_DIM, dim=64)
tokens_vis = demo_embed_vis(sample_tensor)  # (1, 17, 64)

mha_vis = MultiHeadAttention(dim=64, heads=4)
_, attn_matrix = mha_vis(tokens_vis)
# attn_matrix shape: (batch=1, heads=4, tokens=17, tokens=17)

token_labels = ['CLS'] + [f'P{i}' for i in range(16)]

fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'Attention Matrix — 4 Heads × 17 Tokens (CLS + 16 Patches)\n'
    'Row i = "token i attends to..."   Col j = "...token j"\n'
    '(Bright = high attention, weights are pre-training random)',
    fontsize=12, y=1.01
)

for head_idx in range(4):
    ax = fig.add_subplot(2, 3, head_idx + 1)
    attn_np = attn_matrix[0, head_idx].detach().numpy()  # (17, 17)

    im = ax.imshow(attn_np, cmap='Blues', vmin=0, vmax=attn_np.max())
    ax.set_title(f'Head {head_idx + 1}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(17)); ax.set_xticklabels(token_labels, rotation=90, fontsize=7)
    ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=7)
    ax.set_xlabel('Attends TO →', fontsize=9)
    ax.set_ylabel('← FROM token', fontsize=9)

    # Highlight the CLS row with a red border
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.add_patch(plt.Rectangle((-0.5, -0.5), 17, 1,
                                fill=False, edgecolor='red', linewidth=2.5, clip_on=False))
    ax.text(17.2, 0, '← CLS row', color='red', fontsize=8, va='center')

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# ── Bottom-left: CLS row bar chart across all heads ─────────────────────────
ax_bar = fig.add_subplot(2, 3, 5)
width = 0.2
x = np.arange(17)
colors_heads = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']
for h in range(4):
    cls_row = attn_matrix[0, h, 0, :].detach().numpy()
    ax_bar.bar(x + h * width, cls_row, width=width,
               label=f'Head {h+1}', color=colors_heads[h], alpha=0.85)
ax_bar.set_xticks(x + 1.5 * width)
ax_bar.set_xticklabels(token_labels, rotation=90, fontsize=7)
ax_bar.set_title('CLS Token Attention per Head\n(how much CLS looks at each token)', fontsize=10)
ax_bar.set_ylabel('Attention weight')
ax_bar.legend(fontsize=8)
ax_bar.axhline(1/17, color='grey', linestyle='--', linewidth=1, label='Uniform (1/17)')
ax_bar.grid(axis='y', alpha=0.3)

# ── Bottom-right: show the MNIST sample with patch grid ─────────────────────
ax_img = fig.add_subplot(2, 3, 6)
ax_img.imshow(sample_img.squeeze().numpy(), cmap='gray')
n = IMAGE_SIZE // PATCH_SIZE
patch_colors = plt.cm.tab20(np.linspace(0, 1, NUM_PATCHES))
for i in range(n):
    for j in range(n):
        idx = i * n + j
        rect = mpatches.Rectangle(
            (j * PATCH_SIZE - 0.5, i * PATCH_SIZE - 0.5),
            PATCH_SIZE, PATCH_SIZE,
            linewidth=1.5, edgecolor=patch_colors[idx], facecolor='none'
        )
        ax_img.add_patch(rect)
        ax_img.text(j * PATCH_SIZE + PATCH_SIZE // 2 - 0.5,
                    i * PATCH_SIZE + PATCH_SIZE // 2,
                    f'P{idx}', color=patch_colors[idx], fontsize=7,
                    fontweight='bold', ha='center')
ax_img.set_title(f'Input image (label={sample_label})\nwith patch grid', fontsize=10)
ax_img.axis('off')

plt.tight_layout()
plt.show()

print('Note: weights are random (untrained) — after training, CLS will focus on digit strokes.')
print(f'Each row sums to 1.0 (softmax): Head 1 CLS row sum = {attn_matrix[0,0,0,:].sum().item():.4f}')

---
## 第四步：Transformer 塊 / Step 4 — Transformer Block

**中文：** Pre-Norm + 多頭注意力 + FFN（GELU），各帶殘差連接。  
**EN:** Pre-LayerNorm + MHA + FFN (GELU), each with a residual connection.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, mlp_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadAttention(dim, heads, dropout)
        self.ff    = FeedForward(dim, mlp_dim, dropout)

    def forward(self, x):
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ff(x)
        return x, attn_weights


block = TransformerBlock(dim=64, heads=4, mlp_dim=128)
x_in  = torch.randn(2, 17, 64)
x_out, _ = block(x_in)
print(f'輸入 Input : {x_in.shape}')
print(f'輸出 Output: {x_out.shape}  （殘差保形）')

---
### 🔬 Deep Dive — Single-Head Attention Step by Step (with Real Image)

Before stacking everything into a Transformer Block, let's manually trace what happens to all **17 tokens** through one attention head, using a real MNIST image.

**The 4 main steps:**

```
x (17, 64)
   ├─→ × Wq  →  Q (17, 64)   "What am I looking for?"
   ├─→ × Wk  →  K (17, 64)   "What do I contain?"
   └─→ × Wv  →  V (17, 64)   "What I give if attended to"

Scores     = Q @ Kᵀ           (17, 17)  ← every token vs every token
Scaled     = Scores ÷ √64     (17, 17)  ← prevent exploding values
Attention  = softmax(Scaled)  (17, 17)  ← each row sums to 1.0
Output     = Attention @ V    (17, 64)  ← updated token values
```

> `Wq`, `Wk`, `Wv` are **learned weight matrices** — they are what the model trains.

In [ ]:
# ── Single-Head Attention: Step by Step ──────────────────────────────────
torch.manual_seed(0)

# 1. Get a real MNIST image → token sequence
img, label = full_train[0]
with torch.no_grad():
    embed_sh  = PatchEmbedding(patch_dim=PATCH_DIM, dim=64)
    tokens_sh = embed_sh(img.unsqueeze(0))   # (1, 17, 64)
x_sh = tokens_sh[0].detach()                # (17, 64)  remove batch dim

token_labels = ['CLS'] + [f'P{i}' for i in range(16)]
d_k = 64  # single head uses full dim

# 2. Create weight matrices Wq, Wk, Wv
Wq = nn.Linear(64, d_k, bias=False)
Wk = nn.Linear(64, d_k, bias=False)
Wv = nn.Linear(64, d_k, bias=False)

with torch.no_grad():
    Q = Wq(x_sh)                             # (17, 64)
    K = Wk(x_sh)                             # (17, 64)
    V = Wv(x_sh)                             # (17, 64)

    scores_raw    = Q @ K.T                  # (17, 17)  raw dot products
    scores_scaled = scores_raw / (d_k**0.5)  # (17, 17)  ÷ √64 = ÷8
    attn_w_sh     = F.softmax(scores_scaled, dim=-1)  # (17, 17) rows sum=1
    output_sh     = attn_w_sh @ V            # (17, 64)

# ── Plot: 8 steps ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle(f'Single-Head Attention — 8 Steps  (MNIST digit: {label})\n'
             f'17 tokens = 1 CLS + 16 patches,  d_k = {d_k}', fontsize=13)

# Top row: input → Q, K, V
top_items = [
    (x_sh,         'Blues',  f'① Input x\n(17, 64)\noriginal tokens'),
    (Q,            'RdBu_r', f'② Q = x · Wq\n(17, 64)\n"What am I looking for?"'),
    (K,            'RdBu_r', f'③ K = x · Wk\n(17, 64)\n"What do I contain?"'),
    (V,            'RdBu_r', f'④ V = x · Wv\n(17, 64)\n"What I give if chosen"'),
]
for col, (mat, cmap, title) in enumerate(top_items):
    ax = axes[0, col]
    im = ax.imshow(mat.numpy(), cmap=cmap, aspect='auto')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=6)
    ax.set_xlabel('dim (64)', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

# Bottom row: scores → scaled → softmax → output
bot_items = [
    (scores_raw,    'RdBu_r', f'⑤ Scores = Q @ Kᵀ\n(17, 17)\nraw dot products'),
    (scores_scaled, 'RdBu_r', f'⑥ Scaled = ÷ √{d_k}\n(17, 17)\nprevents huge values'),
    (attn_w_sh,     'Blues',  f'⑦ Attention = softmax\n(17, 17)\neach row sums to 1.0'),
    (output_sh,     'RdBu_r', f'⑧ Output = Attn @ V\n(17, 64)\nupdated token values'),
]
for col, (mat, cmap, title) in enumerate(bot_items):
    ax = axes[1, col]
    im = ax.imshow(mat.numpy(), cmap=cmap, aspect='auto')
    ax.set_title(title, fontsize=9, fontweight='bold')
    if mat.shape == (17, 17):
        ax.set_xticks(range(17)); ax.set_xticklabels(token_labels, rotation=90, fontsize=5)
        ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=6)
        if col == 2:  # attention matrix — highlight CLS row
            ax.add_patch(plt.Rectangle((-0.5, -0.5), 17, 1,
                         fill=False, edgecolor='red', linewidth=2.5, clip_on=False))
            ax.text(17.3, 0, '← CLS', color='red', fontsize=7, va='center')
    else:
        ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=6)
        ax.set_xlabel('dim (64)', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

plt.tight_layout()
plt.show()

print('── Shape at each step ──────────────────────────────')
print(f'x input       : {x_sh.shape}')
print(f'Q             : {Q.shape}')
print(f'K             : {K.shape}')
print(f'V             : {V.shape}')
print(f'Q @ Kᵀ        : {scores_raw.shape}  ← 17×17 every pair')
print(f'÷ √{d_k}        : {scores_scaled.shape}')
print(f'softmax       : {attn_w_sh.shape}  row[0] sum = {attn_w_sh[0].sum().item():.4f}')
print(f'Output Attn@V : {output_sh.shape}  ← same shape as input ✓')

---
### 🔀 Extending to Multi-Head Attention (4 Heads)

Single-head uses the **full dim=64** for one set of Q,K,V.  
Multi-head **splits** dim into 4 smaller heads of dim=16 each, runs them in parallel, then **concatenates**.

```
Single-Head:                    Multi-Head (4 heads):
                                
x (17,64)                       x (17,64)
    ↓                            ↙   ↙   ↘   ↘
Q,K,V (17,64)               H1  H2  H3  H4    ← each gets d_head=16
    ↓                        ↓   ↓   ↓   ↓
Attn (17,17)             (17,16)(17,16)(17,16)(17,16)
    ↓                            ↓  concat
Output (17,64)               (17, 64)   ← 4 × 16 = 64
                                 ↓  Linear(64→64)
                             (17, 64)  final output
```

**Why split into heads?**
- Each head learns to focus on **different patterns**
- Head 1 might focus on: nearby patches
- Head 2 might focus on: top vs bottom of image
- Head 3 might focus on: stroke direction
- Head 4 might focus on: background vs ink regions
- Together they capture **richer information** than one big head

In [ ]:
# ── Multi-Head Attention: 4 Heads in Parallel ────────────────────────────
torch.manual_seed(0)

n_heads = 4
d_head  = 64 // n_heads   # 64 ÷ 4 = 16 per head

heads_attn   = []
heads_output = []

with torch.no_grad():
    for h in range(n_heads):
        wq = nn.Linear(64, d_head, bias=False)
        wk = nn.Linear(64, d_head, bias=False)
        wv = nn.Linear(64, d_head, bias=False)

        Q_h     = wq(x_sh)                              # (17, 16)
        K_h     = wk(x_sh)                              # (17, 16)
        V_h     = wv(x_sh)                              # (17, 16)
        score_h = Q_h @ K_h.T / (d_head ** 0.5)        # (17, 17)
        attn_h  = F.softmax(score_h, dim=-1)            # (17, 17)
        out_h   = attn_h @ V_h                          # (17, 16)

        heads_attn.append(attn_h)
        heads_output.append(out_h)

    concat = torch.cat(heads_output, dim=-1)            # (17, 64)  ← 4×16
    W_O    = nn.Linear(64, 64, bias=False)
    final  = W_O(concat)                                # (17, 64)

# ── Figure: 4 attention matrices + CLS row comparison ────────────────────
head_colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']

fig = plt.figure(figsize=(22, 9))
fig.suptitle('Multi-Head Attention — 4 Heads Running in Parallel\n'
             f'Each head: d_head={d_head}  |  All heads combined: (17, 64)', fontsize=12)

# Top row: 4 attention matrices
for h in range(4):
    ax = fig.add_subplot(2, 5, h + 1)
    im = ax.imshow(heads_attn[h].numpy(), cmap='Blues', aspect='auto')
    ax.set_title(f'Head {h+1}\nAttention (17×17)\nd_head={d_head}',
                 fontsize=9, fontweight='bold', color=head_colors[h])
    ax.set_xticks(range(17)); ax.set_xticklabels(token_labels, rotation=90, fontsize=5)
    ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=6)
    ax.add_patch(plt.Rectangle((-0.5, -0.5), 17, 1,
                 fill=False, edgecolor='red', linewidth=2.5, clip_on=False))
    plt.colorbar(im, ax=ax, fraction=0.04)

# Top-right: concat diagram
ax5 = fig.add_subplot(2, 5, 5)
ax5.axis('off')
ax5.set_title('Combining heads', fontsize=10, fontweight='bold')
diagram = (
    'Head 1 → (17, 16)\n'
    'Head 2 → (17, 16)\n'
    'Head 3 → (17, 16)\n'
    'Head 4 → (17, 16)\n'
    '      ↓ concat\n'
    '    (17, 64)\n'
    '      ↓ Linear(64→64)\n'
    '    (17, 64) ✓'
)
ax5.text(0.1, 0.5, diagram, transform=ax5.transAxes, fontsize=10,
         va='center', family='monospace',
         bbox=dict(boxstyle='round', facecolor='#EBF5FB', alpha=0.9))

# Bottom: CLS row attention weight per head (bar chart)
ax_bar = fig.add_subplot(2, 1, 2)
x_pos  = np.arange(17)
width  = 0.2
for h in range(n_heads):
    ax_bar.bar(x_pos + h * width, heads_attn[h][0].numpy(),
               width=width, color=head_colors[h], alpha=0.85, label=f'Head {h+1}')
ax_bar.set_xticks(x_pos + 1.5 * width)
ax_bar.set_xticklabels(token_labels, fontsize=8, rotation=45)
ax_bar.axhline(1/17, color='grey', linestyle='--', linewidth=1.2, label='Uniform (1/17)')
ax_bar.set_title('CLS Token Attention Weight per Head\n'
                 '(notice each head distributes attention differently)', fontsize=11)
ax_bar.set_ylabel('Attention weight')
ax_bar.legend(fontsize=9)
ax_bar.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('── Multi-Head Shape Summary ──────────────────────────')
print(f'Per head  Q/K/V  : (17, {d_head})   ← smaller, specialised')
print(f'Per head  output : (17, {d_head})')
print(f'After concat     : {concat.shape}  ← {n_heads} × {d_head} = 64')
print(f'After W_O proj   : {final.shape}   ← back to original dim')

---
### 🎯 Why Does Output Shape (17, 64) Only Use CLS Token?

After attention, every token has **collected info from all other tokens**.  
But they collected info for **different purposes**:

```
Token    | What it collected                        | Used for?
---------|------------------------------------------|----------
CLS      | Summary of the WHOLE image               | ✅ Classification
P0       | Info about top-left region + neighbours  | ❌ Not used
P1       | Info about top-center region             | ❌ Not used
...      | ...                                      | ❌ Not used
P15      | Info about bottom-right region           | ❌ Not used
```

**CLS was designed with one job: represent the full image.**  
It has no patch of its own, so it is "forced" to gather info from all patches via attention.

After 2 Transformer Blocks:
```
output[:, 0]  → CLS → (batch, 64) → Linear(64→10) → class scores
output[:, 1]  → P0  → discarded
output[:, 2]  → P1  → discarded
...
output[:, 16] → P15 → discarded
```

In [ ]:
# ── Why CLS? — Compare CLS row vs patch rows ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Why Only CLS Token is Used — Output (17, 64) → Take [:, 0]', fontsize=12)

# Left: attention matrix with CLS and P5 rows highlighted
ax = axes[0]
im = ax.imshow(attn_w_sh.numpy(), cmap='Blues', aspect='auto')
ax.set_title('⑦ Attention Matrix\nRow = who is attending FROM\nCol = who is being attended TO', fontsize=9)
ax.set_xticks(range(17)); ax.set_xticklabels(token_labels, rotation=90, fontsize=6)
ax.set_yticks(range(17)); ax.set_yticklabels(token_labels, fontsize=7)
ax.add_patch(plt.Rectangle((-0.5, -0.5), 17, 1,
             fill=False, edgecolor='red', linewidth=3, clip_on=False))
ax.add_patch(plt.Rectangle((-0.5, 5.5),  17, 1,
             fill=False, edgecolor='orange', linewidth=3, clip_on=False))
ax.text(17.3,  0, '← CLS row', color='red',    fontsize=8, va='center', fontweight='bold')
ax.text(17.3,  6, '← P5 row',  color='orange', fontsize=8, va='center', fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)

# Middle: bar chart CLS row vs P5 row
ax = axes[1]
x_pos = np.arange(17)
width = 0.38
ax.bar(x_pos - width/2, attn_w_sh[0].numpy(),  width=width,
       color='red',    alpha=0.8, label='CLS  — attends broadly to all patches')
ax.bar(x_pos + width/2, attn_w_sh[6].numpy(), width=width,
       color='orange', alpha=0.8, label='P5   — attention more concentrated')
ax.axhline(1/17, color='grey', linestyle='--', linewidth=1.2, label='Uniform (1/17)')
ax.set_xticks(x_pos); ax.set_xticklabels(token_labels, rotation=45, fontsize=7)
ax.set_title('CLS attends broadly across all patches\nPatch tokens focus on specific regions', fontsize=10)
ax.set_ylabel('Attention weight')
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3)

# Right: text box showing what gets passed to classifier
ax = axes[2]
ax.axis('off')
ax.set_title('What happens after attention', fontsize=10, fontweight='bold')
text = (
    ' output shape: (17, 64)\n\n'
    ' output[0] = CLS  (64,) ✅\n'
    '    ↓ LayerNorm\n'
    '    ↓ Linear(64 → 10)\n'
    '    [0.01, 0.02, 0.88, ...]\n'
    '       0     1     2  ...\n'
    '                  ↑\n'
    '           predicted digit!\n\n'
    ' output[1]  = P0  (64,) ❌\n'
    ' output[2]  = P1  (64,) ❌\n'
    '    ...             ❌\n'
    ' output[16] = P15 (64,) ❌\n\n'
    ' Patch tokens are discarded\n'
    ' after classification is done.'
)
ax.text(0.05, 0.95, text, transform=ax.transAxes,
        fontsize=10, va='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='#FDFEFE', edgecolor='#AAB7B8', alpha=0.9))

plt.tight_layout()
plt.show()

# Verify row sums
print('── Attention row sums (should all be 1.0) ──')
for i, name in enumerate(token_labels):
    print(f'  {name:4s} row sum: {attn_w_sh[i].sum().item():.4f}')
print(f'\nOutput shape        : {output_sh.shape}')
print(f'CLS token output[0] : {output_sh[0].shape}  → this goes to the classifier')

---
## 第五步：完整模型（小型）/ Step 5 — Full Model (Compact)

**中文：** 小型版 `dim=64, depth=2`，速度快約 10 倍，準確率略降但仍可達 96%+。

```
輸入 (batch, 1, 28, 28)
    ↓ PatchEmbedding  → (batch, 17, 64)
    ↓ + 位置編碼
    ↓ TransformerBlock × 2
    ↓ CLS token x[:,0]  → (batch, 64)
    ↓ LayerNorm → Linear(64, 10) → logits
```

**EN:** Compact version: `dim=64, depth=2` — ~10× faster, ~96% accuracy.

In [ ]:
class MNISTViT(nn.Module):
    def __init__(
        self,
        image_size: int = 28,
        patch_size: int = 7,
        num_classes: int = 10,
        dim: int = 64,        # ⚡ 小型版：64（原版128）
        depth: int = 2,       # ⚡ 小型版：2層（原版6層）
        heads: int = 4,
        mlp_dim: int = 128,   # ⚡ 小型版：128（原版256）
        dropout: float = 0.1,
    ):
        super().__init__()
        assert image_size % patch_size == 0
        num_patches = (image_size // patch_size) ** 2
        patch_dim   = patch_size * patch_size

        self.patch_embed = PatchEmbedding(patch_dim, dim)
        self.pos_embed   = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.dropout     = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim, dropout)
            for _ in range(depth)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        x = self.patch_embed(x)
        x = self.dropout(x + self.pos_embed)

        attn_maps = []
        for block in self.encoder:
            x, attn = block(x)
            attn_maps.append(attn)

        logits = self.classifier(x[:, 0])
        return (logits, attn_maps) if return_attn else logits


model = MNISTViT(
    image_size=28, patch_size=7, num_classes=10,
    dim=64, depth=2, heads=4, mlp_dim=128, dropout=0.1
).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'總參數量   Total parameters    : {total:,}  ⚡（原版約 200k）')
print(f'可訓練參數 Trainable parameters: {trainable:,}')

dummy  = torch.randn(4, 1, 28, 28).to(device)
logits = model(dummy)
print(f'輸出形狀   Output shape        : {logits.shape}  ✓')

---
## 第六步：訓練 / Step 6 — Training

**中文：** Adam + OneCycleLR，訓練 3 個 epoch，10k 資料約 **1–2 分鐘（CPU）**。  
**EN:** Adam + OneCycleLR, 3 epochs on 10k samples — about **1–2 min on CPU**.

> ⚠️ **注意 / Note:** 請先從頭依序執行所有 cell，再執行此 cell。  
> Run all previous cells in order before running this one.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from einops import rearrange, repeat
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

EPOCHS = 10

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3,
    epochs=EPOCHS, steps_per_epoch=len(train_loader)
)
criterion = nn.CrossEntropyLoss()


def evaluate(mdl, loader):
    mdl.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = mdl(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100


train_losses, test_accs = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    acc      = evaluate(model, test_loader)
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f'Epoch {epoch:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(train_losses)+1), train_losses, marker='o', color='steelblue', linewidth=2)
ax1.set_title('訓練損失 / Training Loss', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(test_accs)+1), test_accs, marker='o', color='forestgreen', linewidth=2)
ax2.set_title('測試準確率 / Test Accuracy', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(85, 100)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'MNISTViT Fast — Final Acc: {test_accs[-1]:.2f}%  (10k subset, dim=64, depth=2)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 第七步：混淆矩陣 / Step 7 — Confusion Matrix

**中文：** 對角線 = 正確預測，非對角線 = 錯誤。可看出哪些數字容易搞混（如 4↔9, 3↔8）。  
**EN:** Diagonal = correct, off-diagonal = errors. Shows which digit pairs confuse the model.

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title('混淆矩陣 / Confusion Matrix', fontsize=11)
plt.colorbar(im, ax=axes[0])
tick_marks = np.arange(10)
axes[0].set_xticks(tick_marks); axes[0].set_yticks(tick_marks)
axes[0].set_xticklabels([str(i) for i in range(10)])
axes[0].set_yticklabels([str(i) for i in range(10)])
axes[0].set_xlabel('預測 Predicted')
axes[0].set_ylabel('真實 True')
thresh = cm.max() / 2.0
for i, j in itertools.product(range(10), range(10)):
    axes[0].text(j, i, format(cm[i, j], 'd'), ha='center', va='center', fontsize=8,
                 color='white' if cm[i, j] > thresh else 'black')

per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
bar_colors = ['#E74C3C' if a < 96 else '#2ECC71' for a in per_class_acc]
bars = axes[1].bar(range(10), per_class_acc, color=bar_colors, edgecolor='white', linewidth=1.2)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([f'{i}' for i in range(10)])
axes[1].set_ylim(85, 100.5)
axes[1].set_title('各類別準確率 / Per-class Accuracy', fontsize=11)
axes[1].set_ylabel('Accuracy (%)')
axes[1].axhline(y=np.mean(per_class_acc), color='navy', linestyle='--',
                linewidth=1.5, label=f'Mean: {np.mean(per_class_acc):.2f}%')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle(f'評估結果 / Evaluation — Overall: {test_accs[-1]:.2f}%', fontsize=12)
plt.tight_layout()
plt.show()

---
## 第八步：注意力視覺化 / Step 8 — Attention Visualization

**中文：** CLS 詞元對各塊的注意力 → 模型在看圖像哪個區域？  
**EN:** CLS token attention to each patch — where is the model looking?

---

### 🌡️ What is a Heatmap?

A **heatmap** is a grid where each cell is coloured by its value:

```
Low value  →  Dark / Black
High value →  Bright / White / Yellow
```

Example — attention heatmap for digit "5":
```
┌────┬────┬────┬────┐
│ 🟫 │ 🟫 │ 🟨 │ 🟫 │   Row 0 (top patches)
├────┼────┼────┼────┤
│ 🟫 │ 🟧 │ 🟧 │ 🟫 │   Row 1
├────┼────┼────┼────┤
│ 🟫 │ 🟫 │ 🟧 │ 🟫 │   Row 2
├────┼────┼────┼────┤
│ 🟫 │ 🟨 │ 🟨 │ 🟫 │   Row 3 (bottom patches)
└────┴────┴────┴────┘

🟨 Bright = CLS pays HIGH attention here (important patch)
🟫 Dark   = CLS pays LOW attention here  (background, not useful)
```

In our case:
- The grid is **4×4** (16 patches reshaped from the 16 attention scores)
- Each cell = one patch of the image
- **Bright cell** = the model found this patch important for deciding the digit
- **Dark cell** = the model mostly ignored this patch (likely blank background)

**colormap used:** `'hot'` → black → red → orange → yellow → white (low to high)

---

### Why 3 rows in the visualization?

| Row | Name | What it shows |
|---|---|---|
| Row 1 | Original | Raw grayscale digit image |
| Row 2 | Attention map | 4×4 heatmap of CLS attention scores |
| Row 3 | Overlay | Attention upscaled to 28×28 and blended on top of original |

> Row 3 is the most useful — it shows **exactly which pixels** the model focused on, mapped back onto the original image.

In [ ]:
model.eval()

class_images = {}
for img, label in test_data:
    lbl = label if isinstance(label, int) else label.item()
    if lbl not in class_images:
        class_images[lbl] = img
    if len(class_images) == 10:
        break

fig, axes = plt.subplots(3, 10, figsize=(20, 7))

for col, digit in enumerate(range(10)):
    img = class_images[digit].unsqueeze(0).to(device)

    with torch.no_grad():
        logits, attn_maps = model(img, return_attn=True)
        pred = logits.argmax().item()
        conf = F.softmax(logits, dim=-1)[0, pred].item() * 100

    axes[0, col].imshow(img.cpu().squeeze(), cmap='gray')
    axes[0, col].set_title(
        f'真:{digit} 預:{pred}\n{conf:.1f}%', fontsize=8,
        color='#1E8449' if pred == digit else '#C0392B'
    )
    axes[0, col].axis('off')

    last_attn = attn_maps[-1][0]
    cls_attn  = last_attn[:, 0, 1:].mean(dim=0).cpu().numpy().reshape(4, 4)
    cls_attn  = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)

    axes[1, col].imshow(cls_attn, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'數字 {digit}', fontsize=8)
    axes[1, col].axis('off')

    attn_up = np.kron(cls_attn, np.ones((7, 7)))
    axes[2, col].imshow(img.cpu().squeeze().numpy(), cmap='gray', alpha=0.45)
    axes[2, col].imshow(attn_up, cmap='hot', alpha=0.65)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('原圖', fontsize=8)
axes[1, 0].set_ylabel('注意力圖', fontsize=8)
axes[2, 0].set_ylabel('疊加圖', fontsize=8)
plt.suptitle('CLS 詞元注意力 / CLS Token Attention — Last Layer Mean', fontsize=12)
plt.tight_layout()
plt.show()

---
## 總結 / Summary

### 快速版 vs 完整版 / Fast vs Full

| 設定 Setting | 快速版 Fast ⚡ | 完整版 Full |
|---|---|---|
| 訓練資料 Train data | 10,000 張 | 60,000 張 |
| dim | 64 | 128 |
| depth | 2 | 6 |
| mlp_dim | 128 | 256 |
| 參數量 Params | ~50k | ~200k |
| CPU 訓練時間 | ~1-2 分鐘 | ~10-15 分鐘 |
| 準確率 Accuracy | ~96-97% | ~98-99% |

### 各元件作用 / What Each Component Does

| 元件 | 作用 |
|---|---|
| PatchEmbedding | 把每塊49像素投影成64維向量，加入CLS token |
| 位置編碼 | 告訴模型每塊在圖像哪個位置 |
| Multi-Head Attention | 讓每個token「看」所有其他token |
| FFN | 每個token的非線性特徵變換 |
| CLS token | 收集全圖資訊，最終輸出分類結果 |